# MM-UKAN++ Training for Massachusetts Buildings Segmentation

Semantic segmentation on the Massachusetts Buildings dataset with paired images and labels.

https://github.com/670768312/MM-UKAN-Plus-Plus

In [1]:
import os
import glob
import time
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
PROJECT_ROOT = os.path.abspath("..")
DATASET_ROOT = os.path.join(PROJECT_ROOT, "dataset", "archive")
DATASET_VARIANT = "tiff"  # "tiff" or "png"
MODEL_SAVE_DIR = os.path.join(PROJECT_ROOT, "UKAN++", "models")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
NUM_CLASSES = 2
CLASS_NAMES = ["background", "building"]
IGNORE_INDEX = None  # set to 0 to ignore background in metrics

BATCH_SIZE = 1
ACCUM_STEPS = 8
USE_AMP = torch.cuda.is_available()
LEARNING_RATE = 1e-4
EPOCHS = 50

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train")
TRAIN_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train_labels")
VAL_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val")
VAL_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val_labels")
TEST_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test")
TEST_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test_labels")

def _count_images(path, exts):
    count = 0
    for ext in exts:
        count += len(glob.glob(os.path.join(path, f"*.{ext}")))
    return count

exts = ["tif", "tiff"] if DATASET_VARIANT == "tiff" else ["png", "jpg", "jpeg"]

for p in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR, TEST_IMG_DIR, TEST_MASK_DIR]:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Missing directory: {p}")

print(f"Device: {DEVICE}")
print("Dataset variant:", DATASET_VARIANT)
print("Train images:", _count_images(TRAIN_IMG_DIR, exts), "Train labels:", _count_images(TRAIN_MASK_DIR, exts))
print("Val images:", _count_images(VAL_IMG_DIR, exts), "Val labels:", _count_images(VAL_MASK_DIR, exts))
print("Test images:", _count_images(TEST_IMG_DIR, exts), "Test labels:", _count_images(TEST_MASK_DIR, exts))

Device: cuda
Dataset variant: tiff
Train images: 137 Train labels: 137
Val images: 4 Val labels: 4
Test images: 10 Test labels: 10


## Dataset Loader

In [3]:
class MassachusettsBuildingsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=256, variant="tiff"):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.exts = ["tif", "tiff"] if variant == "tiff" else ["png", "jpg", "jpeg"]

        self.pairs = self._collect_pairs()
        if len(self.pairs) == 0:
            raise RuntimeError(f"No image/mask pairs found in {image_dir} and {mask_dir}")

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.mask_resize = transforms.Resize((img_size, img_size), interpolation=Image.NEAREST)

    def _collect_pairs(self):
        img_paths = []
        for ext in self.exts:
            img_paths.extend(sorted(self.image_dir.glob(f"*.{ext}")))
        mask_map = {}
        for ext in self.exts:
            for p in self.mask_dir.glob(f"*.{ext}"):
                mask_map[p.stem] = p
        pairs = []
        for img_path in img_paths:
            mask_path = mask_map.get(img_path.stem)
            if mask_path is not None:
                pairs.append((img_path, mask_path))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_resize(mask)
        mask = np.array(mask)
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask)
        return image, mask

## MM-UKAN++ Model (from 670768312/MM-UKAN-Plus-Plus)

In [4]:
# Load MM-UKAN-Plus-Plus architecture directly from the GitHub repo.
import os
import sys
import types
import urllib.request

import torch
import torch.nn as nn

MM_UKANPP_SRC = os.path.join(PROJECT_ROOT, "UKAN++", "mm_ukanpp_src")
os.makedirs(MM_UKANPP_SRC, exist_ok=True)


def _download(url, dst, force=False):
    if force or not os.path.exists(dst):
        urllib.request.urlretrieve(url, dst)


_download(
    "https://raw.githubusercontent.com/670768312/MM-UKAN-Plus-Plus/main/archs_1_FCSA_Fusion_concat.py",
    os.path.join(MM_UKANPP_SRC, "archs_1_FCSA_Fusion_concat.py"),
)
_download(
    "https://raw.githubusercontent.com/670768312/MM-UKAN-Plus-Plus/main/kan.py",
    os.path.join(MM_UKANPP_SRC, "kan.py"),
)
_download(
    "https://raw.githubusercontent.com/670768312/MM-UKAN-Plus-Plus/main/FCS_attention.py",
    os.path.join(MM_UKANPP_SRC, "FCS_attention.py"),
)

utils_stub = os.path.join(MM_UKANPP_SRC, "utils.py")
if not os.path.exists(utils_stub):
    with open(utils_stub, "w", encoding="utf-8") as f:
        f.write("class AverageMeter:\n")
        f.write("    def __init__(self):\n")
        f.write("        self.reset()\n")
        f.write("    def reset(self):\n")
        f.write("        self.val = 0\n")
        f.write("        self.avg = 0\n")
        f.write("        self.sum = 0\n")
        f.write("        self.count = 0\n")
        f.write("    def update(self, val, n=1):\n")
        f.write("        self.val = val\n")
        f.write("        self.sum += val * n\n")
        f.write("        self.count += n\n")
        f.write("        self.avg = self.sum / self.count if self.count else 0\n\n")
        f.write("def str2bool(v):\n")
        f.write("    return str(v).lower() in ('true', '1', 'yes', 'y')\n")

try:
    from mmcv.cnn import ConvModule  # noqa: F401
except Exception:
    class ConvModule(nn.Module):
        def __init__(
            self,
            in_channels,
            out_channels,
            kernel_size,
            stride=1,
            padding=0,
            dilation=1,
            groups=1,
            bias=True,
            norm_cfg=None,
            act_cfg=None,
            inplace=True,
        ):
            super().__init__()
            layers = [
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size,
                    stride=stride,
                    padding=padding,
                    dilation=dilation,
                    groups=groups,
                    bias=bias,
                )
            ]
            if norm_cfg is not None:
                layers.append(nn.BatchNorm2d(out_channels))
            if act_cfg is not None:
                act_type = act_cfg.get("type", "ReLU")
                if act_type == "ReLU":
                    layers.append(nn.ReLU(inplace=inplace))
                elif act_type == "LeakyReLU":
                    layers.append(nn.LeakyReLU(0.01, inplace=inplace))
                elif act_type == "SiLU":
                    layers.append(nn.SiLU(inplace=inplace))
            self.block = nn.Sequential(*layers)

        def forward(self, x):
            return self.block(x)

    mmcv = types.ModuleType("mmcv")
    cnn = types.ModuleType("mmcv.cnn")
    cnn.ConvModule = ConvModule
    mmcv.cnn = cnn
    sys.modules["mmcv"] = mmcv
    sys.modules["mmcv.cnn"] = cnn

if MM_UKANPP_SRC not in sys.path:
    sys.path.insert(0, MM_UKANPP_SRC)

from archs_1_FCSA_Fusion_concat import UKAN

/media/aejaz/New_Volume/Projects/Massachusetts/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/aejaz/New_Volume/Projects/Massachusetts/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Losses, Optimizer, and Metrics

In [5]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dims)
        cardinality = torch.sum(probs + targets_one_hot, dims)
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice.mean()


class SegmentationMetrics:
    def __init__(self, num_classes, ignore_index=None, eps=1e-7):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.eps = eps
        self.reset()

    def reset(self):
        self.tp = torch.zeros(self.num_classes, dtype=torch.float64)
        self.fp = torch.zeros(self.num_classes, dtype=torch.float64)
        self.fn = torch.zeros(self.num_classes, dtype=torch.float64)
        self.correct = 0
        self.total = 0

    def update(self, logits, targets):
        preds = torch.argmax(logits, dim=1)
        if self.ignore_index is not None:
            valid_mask = targets != self.ignore_index
            preds = preds[valid_mask]
            targets = targets[valid_mask]
        self.correct += (preds == targets).sum().item()
        self.total += targets.numel()
        for cls in range(self.num_classes):
            pred_cls = preds == cls
            target_cls = targets == cls
            self.tp[cls] += (pred_cls & target_cls).sum().item()
            self.fp[cls] += (pred_cls & ~target_cls).sum().item()
            self.fn[cls] += (~pred_cls & target_cls).sum().item()

    def compute(self):
        precision = self.tp / (self.tp + self.fp + self.eps)
        recall = self.tp / (self.tp + self.fn + self.eps)
        f1 = 2 * precision * recall / (precision + recall + self.eps)
        iou = self.tp / (self.tp + self.fp + self.fn + self.eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + self.eps)
        accuracy = self.correct / max(self.total, 1)
        return {
            "accuracy": float(accuracy),
            "precision": float(precision.mean()),
            "recall": float(recall.mean()),
            "f1": float(f1.mean()),
            "iou": float(iou.mean()),
            "dice": float(dice.mean()),
            "dice_loss": float(1.0 - dice.mean()),
        }


EMBED_DIMS = [128, 160, 256]
model = UKAN(
    num_classes=NUM_CLASSES,
    input_channels=3,
    img_size=IMG_SIZE,
    embed_dims=EMBED_DIMS,
    no_kan=False,
).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

criterion_ce = nn.CrossEntropyLoss()
criterion_dice = DiceLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 7,684,850


/tmp/ipykernel_90156/2720006930.py:74: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## DataLoaders

In [6]:
num_workers = min(4, os.cpu_count() or 1)

train_dataset = MassachusettsBuildingsDataset(
    TRAIN_IMG_DIR,
    TRAIN_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
val_dataset = MassachusettsBuildingsDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
test_dataset = MassachusettsBuildingsDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 137
Val samples: 4
Test samples: 10


In [7]:
def format_metrics(prefix, metrics):
    return (
        f"{prefix} acc={metrics['accuracy']:.4f} "
        f"prec={metrics['precision']:.4f} recall={metrics['recall']:.4f} "
        f"f1={metrics['f1']:.4f} dice={metrics['dice']:.4f} "
        f"iou={metrics['iou']:.4f} dice_loss={metrics['dice_loss']:.4f}"
    )


def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    metrics = SegmentationMetrics(NUM_CLASSES, ignore_index=IGNORE_INDEX)
    total_loss = 0.0
    if is_train:
        optimizer.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(is_train):
        for step, (images, masks) in enumerate(loader):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)
                loss_ce = criterion_ce(logits, masks)
                loss_dice = criterion_dice(logits, masks)
                loss = loss_ce + loss_dice

            if is_train:
                scaled_loss = loss / ACCUM_STEPS
                scaler.scale(scaled_loss).backward()
                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item() * images.size(0)
            metrics.update(logits, masks)
    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, metrics.compute()

## Training Loop

In [8]:
history = {"train_loss": [], "val_loss": [], "train_metrics": [], "val_metrics": []}
best_val_loss = float("inf")
best_model_path = os.path.join(MODEL_SAVE_DIR, "ukanpp_best.pth")
train_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_train_val.csv")
best_train_loss = float("inf")
best_train_epoch = 0

train_start = time.time()
print("Training start:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_start)))

with open(train_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["epoch", "split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        train_loss, train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
        val_loss, val_metrics = run_epoch(val_loader, model, optimizer=None)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_metrics"].append(train_metrics)
        history["val_metrics"].append(val_metrics)

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_train_epoch = epoch + 1

        writer.writerow([
            epoch + 1,
            "train",
            train_loss,
            train_metrics["accuracy"],
            train_metrics["precision"],
            train_metrics["recall"],
            train_metrics["f1"],
            train_metrics["dice"],
            train_metrics["iou"],
            train_metrics["dice_loss"],
        ])
        writer.writerow([
            epoch + 1,
            "val",
            val_loss,
            val_metrics["accuracy"],
            val_metrics["precision"],
            val_metrics["recall"],
            val_metrics["f1"],
            val_metrics["dice"],
            val_metrics["iou"],
            val_metrics["dice_loss"],
        ])
        metrics_file.flush()

        print(f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
        print(format_metrics("Train", train_metrics))
        print(format_metrics("Val", val_metrics))
        print(f"Epoch time: {time.time() - epoch_start:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                },
                best_model_path,
            )
            print(f"Saved best model: {best_model_path}")

train_end = time.time()
print("Training end:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_end)))
print(f"Total training time: {train_end - train_start:.1f}s")
print(f"Best train epoch: {best_train_epoch} (train_loss={best_train_loss:.4f})")
print(f"Train/val metrics saved to {train_metrics_path}")

Training start: 2026-05-29 20:36:00
Epoch 1/50 | train_loss=1.2468 val_loss=1.2539
Train acc=0.5924 prec=0.5132 recall=0.5275 f1=0.4727 dice=0.4727 iou=0.3460 dice_loss=0.5273
Val acc=0.5970 prec=0.5217 recall=0.5540 f1=0.4707 dice=0.4707 iou=0.3463 dice_loss=0.5293
Epoch time: 7.4s
Saved best model: /media/aejaz/New_Volume/Projects/Massachusetts/UKAN++/models/ukanpp_best.pth
Epoch 2/50 | train_loss=1.2219 val_loss=1.2283
Train acc=0.6290 prec=0.5286 recall=0.5576 f1=0.5004 dice=0.5004 iou=0.3729 dice_loss=0.4996
Val acc=0.6297 prec=0.5337 recall=0.5814 f1=0.4948 dice=0.4948 iou=0.3699 dice_loss=0.5052
Epoch time: 7.1s
Saved best model: /media/aejaz/New_Volume/Projects/Massachusetts/UKAN++/models/ukanpp_best.pth
Epoch 3/50 | train_loss=1.1990 val_loss=1.2114
Train acc=0.6559 prec=0.5415 recall=0.5811 f1=0.5219 dice=0.5219 iou=0.3940 dice_loss=0.4781
Val acc=0.6470 prec=0.5414 recall=0.5987 f1=0.5089 dice=0.5089 iou=0.3835 dice_loss=0.4911
Epoch time: 7.0s
Saved best model: /media/aejaz

## Test Evaluation

In [9]:
if os.path.isfile(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from {best_model_path}")

test_loss, test_metrics = run_epoch(test_loader, model, optimizer=None)
print(f"Test loss: {test_loss:.4f}")
print(format_metrics("Test", test_metrics))

test_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_test.csv")
with open(test_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])
    writer.writerow([
        "test",
        test_loss,
        test_metrics["accuracy"],
        test_metrics["precision"],
        test_metrics["recall"],
        test_metrics["f1"],
        test_metrics["dice"],
        test_metrics["iou"],
        test_metrics["dice_loss"],
    ])
print(f"Test metrics saved to {test_metrics_path}")

Loaded best model from /media/aejaz/New_Volume/Projects/Massachusetts/UKAN++/models/ukanpp_best.pth


/tmp/ipykernel_90156/1889342327.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=DEVICE)


Test loss: 0.9347
Test acc=0.7888 prec=0.5993 recall=0.5553 f1=0.5608 dice=0.5608 iou=0.4602 dice_loss=0.4392
Test metrics saved to /media/aejaz/New_Volume/Projects/Massachusetts/UKAN++/models/metrics_test.csv


## Save Final Model

In [10]:
final_model_path = os.path.join(MODEL_SAVE_DIR, "ukanpp_final.pth")
torch.save(model.state_dict(), final_model_path)
print(f"Final model saved to {final_model_path}")

Final model saved to /media/aejaz/New_Volume/Projects/Massachusetts/UKAN++/models/ukanpp_final.pth
